In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from collections import OrderedDict
import warnings, math, random

# Paths
BAG_YEAR_30M = Path("/mnt/intenso/masterarbeit/landsat/BAG_year_30m_tau050.tif")
CAY_DIR = Path("/mnt/intenso/masterarbeit/landsat/tsa_full/CAY")
CAY_GLOB = "*_CAY.tif"  # e.g., MAX_NDV_CAY.tif etc.
EVAL_MASK_30M = None    # optional Path to 30 m boolean/0-1 mask; or keep None

# Valid year bounds
MIN_YEAR = 1982
MAX_YEAR = 2023

# Sampling
MAX_SAMPLES = 200_000      # cap to keep memory reasonable
MIN_FEATURES_REQUIRED = 1  # require at least this many non-NaN CAY values
RANDOM_SEED = 42

# Spatial split blocks (in pixels on 30 m grid)
BLOCK_H = 128
BLOCK_W = 128

# RF params (tune later)
RF_PARAMS = dict(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    oob_score=False,
)
warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


In [2]:
def read_band(path: Path):
    ds = rio.open(path)
    arr = ds.read(1)
    return ds, arr

# Target
bag_ds, y_arr = read_band(BAG_YEAR_30M)
H, W = bag_ds.height, bag_ds.width
transform = bag_ds.transform
crs = bag_ds.crs

# Valid target mask
y_valid = (y_arr >= MIN_YEAR) & (y_arr <= MAX_YEAR)

# Optional external evaluation mask
if EVAL_MASK_30M is not None:
    m_ds, m_arr = read_band(EVAL_MASK_30M)
    same = (m_ds.crs == crs and m_ds.transform == transform and m_ds.width == W and m_ds.height == H)
    if not same:
        m_re = np.zeros((H, W), dtype=m_arr.dtype)
        reproject(
            source=m_arr, destination=m_re,
            src_transform=m_ds.transform, src_crs=m_ds.crs,
            dst_transform=transform, dst_crs=crs,
            resampling=Resampling.nearest,
            src_nodata=0, dst_nodata=0
        )
        m_arr = m_re
    eval_mask = (m_arr > 0) & y_valid
else:
    eval_mask = y_valid

# CAY rasters
cay_paths = sorted(CAY_DIR.glob(CAY_GLOB))
assert len(cay_paths) > 0, f"No CAY rasters found in {CAY_DIR}"
print("CAY features:", [p.name for p in cay_paths])


CAY features: ['MAX_NDB_CAY.tif', 'MAX_NDV_CAY.tif', 'MAX_NDW_CAY.tif', 'MAX_TCB_CAY.tif', 'MAX_TCW_CAY.tif', 'MIN_NDB_CAY.tif', 'MIN_NDV_CAY.tif', 'MIN_NDW_CAY.tif', 'MIN_TCB_CAY.tif', 'MIN_TCW_CAY.tif', 'Q90_NDB_CAY.tif', 'Q90_NDV_CAY.tif', 'Q90_NDW_CAY.tif', 'Q90_TCB_CAY.tif', 'Q90_TCW_CAY.tif']


In [3]:
# We need pixels where target is valid and at least one feature is nonzero/valid.
# To avoid loading all features, we’ll first pick a candidate set uniformly from eval_mask.
ys, xs = np.where(eval_mask)
n_candidates = ys.size
print(f"Eval mask pixels: {n_candidates:,}")

if n_candidates == 0:
    raise RuntimeError("No valid evaluation pixels.")

if n_candidates > MAX_SAMPLES:
    idx_sel = np.random.choice(n_candidates, size=MAX_SAMPLES, replace=False)
    ys = ys[idx_sel]; xs = xs[idx_sel]
    print(f"Subsampled to {ys.size:,} pixels for modeling.")
else:
    print("Using all valid pixels (may be large).")


Eval mask pixels: 71,028
Using all valid pixels (may be large).


In [4]:
# Use rasterio.sample to read feature values at (x, y) locations for each CAY raster
# Convert pixel indices to map coords: col,row -> x,y
xs_map = transform * (xs + 0.5, ys + 0.5)  # Affine maps (col,row) to (x,y); adding 0.5 centers in pixel

# Build feature matrix
X_list = []
feature_names = []
for p in cay_paths:
    ds = rio.open(p)
    # Quick grid check; reproject if needed (shouldn't be necessary in your setup)
    same = (ds.crs == crs and ds.transform == transform and ds.width == W and ds.height == H)
    if same:
        # Fast path: direct indexing
        arr = ds.read(1)
        vals = arr[ys, xs]
    else:
        # Safe path: use sampling in map coords
        vals = np.fromiter((v[0] for v in ds.sample(zip(*xs_map))), dtype=np.float32, count=ys.size)

    # Treat out-of-range or 0 as missing
    vals = vals.astype(np.float32)
    bad = (vals < MIN_YEAR) | (vals > MAX_YEAR)
    vals[bad] = np.nan

    X_list.append(vals)
    feature_names.append(p.stem)

# Stack to [n_samples, n_features]
X = np.vstack(X_list).T
y = y_arr[ys, xs].astype(np.float32)

# Require at least MIN_FEATURES_REQUIRED non-NaN features
feat_count = np.sum(~np.isnan(X), axis=1)
keep = feat_count >= MIN_FEATURES_REQUIRED
X = X[keep]; y = y[keep]; ys = ys[keep]; xs = xs[keep]
print(f"Kept samples after feature-availability filter: {X.shape[0]:,}")

# Optional: add simple derived features (agreement statistics)
# Example: per-row finite stats (min/max/median/std across indices)
finite = np.where(np.isnan(X), np.nan, X)
row_min = np.nanmin(finite, axis=1)
row_max = np.nanmax(finite, axis=1)
row_med = np.nanmedian(finite, axis=1)
row_std = np.nanstd(finite, axis=1)
X = np.column_stack([X, row_min, row_max, row_med, row_std])
feature_names = feature_names + ["agg_min", "agg_max", "agg_median", "agg_std"]
print("Final feature count:", len(feature_names))


Kept samples after feature-availability filter: 7,278
Final feature count: 19


In [5]:
# Block IDs derived from pixel coordinates to reduce spatial leakage
block_id = (ys // BLOCK_H) * math.ceil(W / BLOCK_W) + (xs // BLOCK_W)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=block_id))

X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

groups_train = block_id[train_idx]
groups_test  = block_id[test_idx]

print(f"Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}")


Train: 5,849  Test: 1,429


In [ ]:
# Pipeline: median impute NaNs -> RandomForest
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(**RF_PARAMS),
) 

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
bias = float(np.mean(y_pred - y_test))

print(f"MAE:  {mae:.2f} years")
print(f"RMSE: {rmse:.2f} years")
print(f"R^2:  {r2:.3f}")
print(f"Bias: {bias:+.2f} years")

# Feature importance (Gini)
rf = model.named_steps["randomforestregressor"]
gini_imp = rf.feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": gini_imp}).sort_values("importance", ascending=False)
display(imp_df.head(20))


MAE:  10.27 years
RMSE: 3.50 years
R^2:  -0.154
Bias: -1.97 years


,feature,importance
17,agg_median,0.135856
16,agg_max,0.124881
15,agg_min,0.123519
18,agg_std,0.056580
12,Q90_NDW_CAY,0.055550
7,MIN_NDW_CAY,0.046275
13,Q90_TCB_CAY,0.045989
2,MAX_NDW_CAY,0.044243
11,Q90_NDV_CAY,0.043544
8,MIN_TCB_CAY,0.041213


In [10]:
perm = permutation_importance(
    model, X_test, y_test, n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1
)
perm_df = pd.DataFrame({
    "feature": feature_names,
    "perm_importance_mean": perm.importances_mean,
    "perm_importance_std": perm.importances_std,
}).sort_values("perm_importance_mean", ascending=False)
display(perm_df.head(20))


,feature,perm_importance_mean,perm_importance_std
14,Q90_TCW_CAY,0.001896,0.000434
9,MIN_TCW_CAY,-0.001221,0.001209
13,Q90_TCB_CAY,-0.001411,0.003305
2,MAX_NDW_CAY,-0.003055,0.000397
4,MAX_TCW_CAY,-0.003875,0.000458
3,MAX_TCB_CAY,-0.004270,0.002548
8,MIN_TCB_CAY,-0.004389,0.003822
0,MAX_NDB_CAY,-0.005608,0.002618
10,Q90_NDB_CAY,-0.005647,0.000747
1,MAX_NDV_CAY,-0.006640,0.002157
